In [25]:
!pip install requests pytest -q

In [30]:
import requests

# National Weather Service API
NWS_BASE_URL = "https://api.weather.gov"

HEADERS = {
    "User-Agent": "FriendlyWeatherChecker/1.0 (your-email@example.com)",
    "Accept": "application/geo+json"
}

# Supported cities
CITIES = {
    "Denver": (39.7392, -104.9903),
    "Washington": (38.9072, -77.0369),
    "Chantilly": (38.8943, -77.4311),
    "Manhattan": (40.7831, -73.9712),
    "Los Angeles": (34.0522, -118.2437)
}


# TOOL: Retrieve weather from the NWS API
def get_weather(city):

    # Match city names regardless of capitalization
    city = city.strip()

    matched_city = next(
        (name for name in CITIES
         if name.lower() == city.lower()),
        None
    )

    if matched_city is None:
        raise ValueError(
            f"Unsupported city: {city}. "
            f"Choose from: {', '.join(CITIES.keys())}"
        )

    latitude, longitude = CITIES[matched_city]

    # Step 1: Find nearby NWS observation stations
    point_url = f"{NWS_BASE_URL}/points/{latitude},{longitude}"

    response = requests.get(
        point_url,
        headers=HEADERS,
        timeout=15
    )
    response.raise_for_status()

    stations_url = response.json()["properties"]["observationStations"]

    stations_response = requests.get(
        stations_url,
        headers=HEADERS,
        timeout=15
    )
    stations_response.raise_for_status()

    stations = stations_response.json()["features"]

    if not stations:
        raise ValueError(
            f"No weather stations found near {matched_city}."
        )

    # Step 2: Find a station with available observations
    data = None

    for station in stations[:5]:
        station_id = station["properties"]["stationIdentifier"]

        observation_url = (
            f"{NWS_BASE_URL}/stations/"
            f"{station_id}/observations/latest"
        )

        try:
            weather_response = requests.get(
                observation_url,
                headers=HEADERS,
                timeout=15
            )
            weather_response.raise_for_status()

            observation = weather_response.json()["properties"]

            if (
                (observation.get("temperature") or {}).get("value")
                is not None
            ):
                data = observation
                break

        except requests.RequestException:
            continue

    if data is None:
        raise ValueError(
            f"No usable weather observations found near {matched_city}."
        )

    # Step 3: Extract weather measurements
    def get_value(field):
        return (data.get(field) or {}).get("value")

    # Convert temperature from Celsius to Fahrenheit
    temperature = get_value("temperature")

    if temperature is not None:
        temperature = round((temperature * 9 / 5) + 32, 1)

    # Convert precipitation from meters to inches
    precipitation = get_value("precipitationLastHour")

    if precipitation is not None:
        precipitation = round(precipitation * 39.3701, 3)

    # Relative humidity (%)
    humidity = get_value("relativeHumidity")

    # Return only the requested measurements
    return {
        "temperature": temperature,
        "precipitation": precipitation,
        "humidity": humidity
    }

Retrieves data for any US city.

National wx service api

In [35]:
def weather_agent(city):

    # Retrieve weather from the NWS API
    weather = get_weather(city)

    temperature = weather["temperature"]
    precipitation = weather["precipitation"]
    humidity = weather["humidity"]

    alerts = []

    # Temperature alerts
    if temperature is not None:
        if temperature >= 100:
            alerts.append(
                "EXTREME HEAT ALERT: Limit outdoor activities."
            )
        elif temperature <= 32:
            alerts.append(
                "FREEZING TEMPERATURE ALERT: Dress warmly."
            )

    # Precipitation alerts
    if precipitation is not None:
        if precipitation >= 0.5:
            alerts.append(
                "HEAVY PRECIPITATION ALERT: Use caution outdoors."
            )
        elif precipitation > 0:
            alerts.append(
                "PRECIPITATION NOTICE: Rain or snow may be occurring."
            )

    # Humidity alerts
    if humidity is not None:
        if humidity >= 85:
            alerts.append(
                "HIGH HUMIDITY NOTICE: Conditions may feel uncomfortable."
            )
        elif humidity <= 20:
            alerts.append(
                "LOW HUMIDITY NOTICE: Dry conditions detected."
            )

    # Format unavailable measurements
    def format_value(value, unit):
        if value is None:
            return "Not available"
        return f"{value} {unit}"

    # Generate weather summary
    summary = f"""
========================================
    FRIENDLY WEATHER NEIGHBORHOOD CHECKER
========================================

Weather Report for {city}

CURRENT CONDITIONS
----------------------------------------
Temperature: {format_value(temperature, '°F')}
Precipitation (Last Hour): {format_value(precipitation, 'inches')}
Humidity: {format_value(humidity, '%')}

WEATHER ALERTS
----------------------------------------
{chr(10).join(alerts) if alerts else "No weather concerns detected by the agent's rules."}

========================================
"""

    return summary

In [ ]:
def start_weather_agent():
    print("Hello, I am your friendly weather neighborhood checker.")
    print("What can I help you with today?")

    while True:
        city = input(
            "\nWhich city would you like the weather for? "
        ).strip()

        if city.lower() in ["exit", "quit", "bye"]:
            print("Thank you for using your friendly weather neighborhood checker!")
            print("Stay safe and have a wonderful day!")
            break

        # Match city names regardless of capitalization
        matched_city = next(
            (name for name in CITIES
             if name.lower() == city.lower()),
            None
        )

        if matched_city is None:
            print("Sorry, I don't recognize that city.")
            print("Available cities:", ", ".join(CITIES.keys()))
            continue

        try:
            print(f"\nChecking the weather in {matched_city}...")
            print(weather_agent(matched_city))

        except Exception as error:
            print(f"Sorry, I couldn't retrieve the weather: {error}")

        print("\nIs there another city I can check for you?")

# Start the interactive weather agent
start_weather_agent()

Hello, I am your friendly weather neighborhood checker.
What can I help you with today?

Which city would you like the weather for? Denver

Checking the weather in Denver...

    FRIENDLY WEATHER NEIGHBORHOOD CHECKER

Weather Report for Denver

CURRENT CONDITIONS
----------------------------------------
Temperature: 42.8 °F
Precipitation (Last Hour): Not available
Humidity: 100 %

WEATHER ALERTS
----------------------------------------
HIGH HUMIDITY NOTICE: Conditions may feel uncomfortable.



Is there another city I can check for you?


User Prompts